## Setup

In [1]:
import importlib
from pathlib import Path
import sys
import polars as pl

pl.Config.set_tbl_rows(25)

REPO_DIR = Path('.').resolve().parent
DEV_DATA_DIR = REPO_DIR / 'trio_dev_data'

sys.path.insert(0, str(REPO_DIR / 'src'))
sys.path.insert(0, str(REPO_DIR / 'src' / 'util'))

In [3]:
KID_ID = 'NA12878'
DAD_ID = 'NA12891'
MOM_ID = 'NA12892'

METH_BASE_DIR = DEV_DATA_DIR / 'output' / 'dna-methylation'

# Unphased (combined) methylation BEDs from aligned_bam_to_cpg_scores
METH_COUNT_DIR = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.count.pedmec-phased' # output dir of aligned_bam_to_cpg_scores (containing count-based unphased meth)
METH_MODEL_DIR = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.model.pedmec-phased' # output dir of aligned_bam_to_cpg_scores (containing model-based unphased meth)

def combined_meth_paths(uid):
    """Return (count_combined, model_combined) paths for a sample."""
    return (
        str(METH_COUNT_DIR / f'{uid}.GRCh38.haplotagged.combined.bed.gz'), # unphased count-based meth from aligned_bam_to_cpg_scores
        str(METH_MODEL_DIR / f'{uid}.GRCh38.haplotagged.combined.bed.gz'), # unphased model-based meth from aligned_bam_to_cpg_scores
    )

KID_METH_COUNT_COMBINED, KID_METH_MODEL_COMBINED = combined_meth_paths(KID_ID)
DAD_METH_COUNT_COMBINED, DAD_METH_MODEL_COMBINED = combined_meth_paths(DAD_ID)
MOM_METH_COUNT_COMBINED, MOM_METH_MODEL_COMBINED = combined_meth_paths(MOM_ID)

# Parent-phased methylation BED
PARENT_PHASED_DIR = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.parent-phased' # output dir of phase_meth_to_parent_haps.py
BED_METH_PARENT_PHASED = str(PARENT_PHASED_DIR / 'trio.dna-methylation.bed') # parent-phased methylation levels from phase_meth_to_parent_haps.py

# All CpG sites in the reference genome (shared with pedigree workflow)
OUTPUT_DIR = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.parent-phased.all-cpgs'
BED_ALL_CPGS_IN_REFERENCE = str(DEV_DATA_DIR / 'output' / 'dna-methylation' / 'CEPH1463.GRCh38.hifi.founder-phased.all-cpgs' / 'all_cpg_sites_in_reference.bed') # output of write_all_cpgs_in_reference.py

## Get all CpG sites in reference genome

In [4]:
import expand_to_all_cpgs_trio
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import read_all_cpgs_in_reference

DF_ALL_CPGS_IN_REFERENCE = read_all_cpgs_in_reference(BED_ALL_CPGS_IN_REFERENCE)
DF_ALL_CPGS_IN_REFERENCE

FileNotFoundError: No such file or directory (os error 2): ...-methylation/CEPH1463.GRCh38.hifi.founder-phased.all-cpgs/all_cpg_sites_in_reference.bed (set POLARS_VERBOSE=1 to see full path)

## Read in unphased DNA methylation at CpG sites for kid, dad, and mom

In [ ]:
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import read_meth_unphased_trio

DF_METH_UNPHASED = read_meth_unphased_trio(
    bed_meth_count_kid=KID_METH_COUNT_COMBINED,
    bed_meth_model_kid=KID_METH_MODEL_COMBINED,
    bed_meth_count_dad=DAD_METH_COUNT_COMBINED,
    bed_meth_model_dad=DAD_METH_MODEL_COMBINED,
    bed_meth_count_mom=MOM_METH_COUNT_COMBINED,
    bed_meth_model_mom=MOM_METH_MODEL_COMBINED,
)
DF_METH_UNPHASED

## Read in parent-phased DNA methylation at CpG sites

In [ ]:
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import read_meth_parent_phased

DF_METH_PARENT_PHASED = read_meth_parent_phased(BED_METH_PARENT_PHASED)
DF_METH_PARENT_PHASED